# USEReady AI/ML Internship Assignment 2
# Power System Load Type Prediction

**Objective:** Predict `Load_Type` (`Light_Load`, `Medium_Load`, or `Maximum_Load`) from historical power-system measurements.

This notebook follows the supplied problem statement:
- Data preprocessing
- Exploratory data analysis (EDA)
- Feature engineering
- Model selection
- Use the **last month of chronological data** as the final test set
- Evaluate using accuracy, precision, recall, and F1-score

> Run the notebook from top to bottom. It expects `load_data.csv` in the same directory as this notebook. If it is elsewhere, update `DATA_PATH` in the next cell.


## 1. Imports and configuration

The models are evaluated with a chronological validation split from the training period. The final month is kept untouched until the final evaluation.


In [ ]:
# If needed, install dependencies in a notebook cell:
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
DATA_PATH = Path("load_data.csv")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

print("Libraries imported.")
print("Notebook directory:", Path.cwd())
print("Dataset path:", DATA_PATH.resolve())


## 2. Load and inspect the dataset

The code validates the required target and parses `Date_Time`. It also tolerates minor differences in column-name whitespace.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place load_data.csv beside this notebook or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))


## 3. Data cleaning and time ordering

Rows without the target label cannot be used for supervised training and are removed. Dates are parsed, invalid timestamps are excluded, and rows are sorted chronologically. Duplicate rows are reported and removed.


In [ ]:
TARGET = "Load_Type"
DATETIME_COL = "Date_Time"

if TARGET not in df.columns:
    raise ValueError(f"Required target column {TARGET!r} was not found.")

if DATETIME_COL not in df.columns:
    raise ValueError(f"Required datetime column {DATETIME_COL!r} was not found.")

df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce", dayfirst=True)

before = len(df)
df = df.dropna(subset=[DATETIME_COL, TARGET]).copy()
print(f"Removed {before - len(df)} rows with invalid/missing datetime or target.")

# Normalize target labels while preserving the assignment's class names.
df[TARGET] = df[TARGET].astype(str).str.strip()
print("Target labels:", sorted(df[TARGET].unique().tolist()))

print("Duplicate rows before removal:", int(df.duplicated().sum()))
df = df.drop_duplicates().sort_values(DATETIME_COL).reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Date range:", df[DATETIME_COL].min(), "to", df[DATETIME_COL].max())
display(df.head())


## 4. Exploratory data analysis (EDA)

Inspect the target distribution, numeric summaries, time coverage, and correlations. These plots help understand the data before modeling.


In [ ]:
plt.figure(figsize=(8, 4))
order = df[TARGET].value_counts().index
sns.countplot(data=df, x=TARGET, order=order)
plt.title("Distribution of Load_Type")
plt.xlabel("Load_Type")
plt.ylabel("Number of records")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(df[TARGET].value_counts().rename("count").to_frame())
display(df.describe(include="all").T)

monthly_counts = df.set_index(DATETIME_COL).resample("MS").size()
plt.figure(figsize=(12, 4))
monthly_counts.plot()
plt.title("Records per Month")
plt.xlabel("Month")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

numeric_cols_for_eda = df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_cols_for_eda) >= 2:
    plt.figure(figsize=(11, 7))
    sns.heatmap(df[numeric_cols_for_eda].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Numeric Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for a correlation matrix.")


## 5. Feature engineering

Create calendar/time features from `Date_Time`. Keep the original timestamp for the chronological split, but do not pass it directly to the estimator. The `NSM` column is retained if present.


In [ ]:
def add_time_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    dt = out[DATETIME_COL]
    out["year"] = dt.dt.year
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["day_of_week"] = dt.dt.dayofweek
    out["day_of_year"] = dt.dt.dayofyear
    out["hour"] = dt.dt.hour
    out["minute"] = dt.dt.minute
    out["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)
    # Cyclic features can represent time-of-day/month continuity.
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out

df_model = add_time_features(df)

# Drop raw datetime and target from predictors.
X_all = df_model.drop(columns=[TARGET, DATETIME_COL])
y_all = df_model[TARGET].astype(str)

# Keep only columns usable by sklearn. Numeric columns are imputed in the pipeline.
# A categorical feature, if present, will be one-hot encoded.
numeric_features = X_all.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_all.columns if c not in numeric_features]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Class labels:", sorted(y_all.unique().tolist()))


## 6. Chronological train/test split

**Required strategy:** the last calendar month in the dataset is the final test set. Earlier data is used for training/model selection. This avoids randomly mixing future observations into training.

If the last month contains too few observations, inspect the printed counts and the dataset's date coverage before proceeding.


In [ ]:
last_timestamp = df_model[DATETIME_COL].max()
test_month_start = last_timestamp.to_period("M").start_time
test_mask = df_model[DATETIME_COL] >= test_month_start

train_df = df_model.loc[~test_mask].copy()
test_df = df_model.loc[test_mask].copy()

if train_df.empty or test_df.empty:
    raise ValueError(
        "Chronological split produced an empty train or test set. "
        "Check Date_Time parsing and dataset coverage."
    )

X_train_all = train_df.drop(columns=[TARGET, DATETIME_COL])
y_train_all = train_df[TARGET].astype(str)
X_test = test_df.drop(columns=[TARGET, DATETIME_COL])
y_test = test_df[TARGET].astype(str)

print("Last-month test period starts:", test_month_start)
print("Training date range:", train_df[DATETIME_COL].min(), "to", train_df[DATETIME_COL].max())
print("Test date range:", test_df[DATETIME_COL].min(), "to", test_df[DATETIME_COL].max())
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("\nTraining target distribution:")
display(y_train_all.value_counts().to_frame("count"))
print("\nTest target distribution:")
display(y_test.value_counts().to_frame("count"))

if train_df[DATETIME_COL].max() >= test_df[DATETIME_COL].min():
    raise AssertionError("Chronological split leakage detected.")


## 7. Preprocessing and candidate models

Numeric features are median-imputed and standardized for logistic regression. Categorical features, if any, are imputed and one-hot encoded. Tree-based models use the same safe preprocessing pipeline.

The majority-class baseline provides a reference point. Models are compared using a chronological validation set from the pre-test training data; the final month remains untouched until Section 9.


In [ ]:
# Create a chronological validation split from the pre-test training period.
if len(train_df) < 10:
    raise ValueError("Too few pre-test rows to create a meaningful validation split.")

validation_month_start = train_df[DATETIME_COL].max().to_period("M").start_time
fit_mask = train_df[DATETIME_COL] < validation_month_start
val_mask = train_df[DATETIME_COL] >= validation_month_start

# If the final training month is too small or would leave no fit rows,
# fall back to the final 20% of chronological rows for validation.
if fit_mask.sum() == 0 or val_mask.sum() == 0:
    split_idx = max(1, int(len(train_df) * 0.8))
    fit_part = train_df.iloc[:split_idx].copy()
    val_part = train_df.iloc[split_idx:].copy()
else:
    fit_part = train_df.loc[fit_mask].copy()
    val_part = train_df.loc[val_mask].copy()

if fit_part.empty or val_part.empty:
    raise ValueError("Could not create a non-empty chronological validation split.")

X_fit = fit_part.drop(columns=[TARGET, DATETIME_COL])
y_fit = fit_part[TARGET].astype(str)
X_val = val_part.drop(columns=[TARGET, DATETIME_COL])
y_val = val_part[TARGET].astype(str)

numeric_features = X_fit.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_fit.columns if c not in numeric_features]

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocess, numeric_features),
        ("categorical", categorical_preprocess, categorical_features)
    ],
    remainder="drop"
)

models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced_subsample",
        random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=250, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
    )
}

validation_results = []
fitted_validation_models = {}

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", clone(estimator))
    ])
    pipe.fit(X_fit, y_fit)
    pred = pipe.predict(X_val)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "precision_macro": precision_score(y_val, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_val, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_val, pred, average="macro", zero_division=0)
    }
    validation_results.append(row)
    fitted_validation_models[name] = pipe

validation_scores = pd.DataFrame(validation_results).sort_values(
    by=["f1_macro", "accuracy"], ascending=False
).reset_index(drop=True)

display(validation_scores)
print("Validation period:", val_part[DATETIME_COL].min(), "to", val_part[DATETIME_COL].max())


## 8. Select the best model and refit on all pre-test data

The model with the best validation macro-F1 is selected. It is then refit using all data before the final month. No final-test labels are used during model selection.


In [ ]:
best_model_name = validation_scores.iloc[0]["model"]
best_estimator = clone(models[best_model_name])

final_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", best_estimator)
])

final_model.fit(X_train_all, y_train_all)
print("Selected model:", best_model_name)
print("Refit rows:", len(X_train_all))


## 9. Final test evaluation (last month)

This is the main final evaluation requested in the assignment. Report accuracy, macro precision, macro recall, macro F1, and a per-class classification report. The confusion matrix shows which load categories are confused.


In [ ]:
y_test_pred = final_model.predict(X_test)

final_metrics = {
    "model": best_model_name,
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision_macro": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_test_pred, average="macro", zero_division=0)
}

print("Final test period:", test_df[DATETIME_COL].min(), "to", test_df[DATETIME_COL].max())
print("Test rows:", len(y_test))
display(pd.DataFrame([final_metrics]).set_index("model"))

print("\nClassification report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, ax=ax, cmap="Blues", xticks_rotation=30
)
ax.set_title(f"Final Test Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 10. Feature importance (tree models)

For tree-based models, show model feature importance after preprocessing. If the selected model does not expose feature importances, use permutation importance on the held-out test set as a diagnostic. The test set is used here only for post-selection analysis; do not use these results to tune the model.


In [ ]:
model_step = final_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    try:
        feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
        importances = pd.Series(model_step.feature_importances_, index=feature_names).sort_values(ascending=False).head(20)
        display(importances.rename("importance").to_frame())
        plt.figure(figsize=(10, 6))
        importances.sort_values().plot(kind="barh")
        plt.title("Top 20 Feature Importances")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("Could not display built-in feature importances:", e)
else:
    print(f"{best_model_name} does not expose built-in feature_importances_.")
    print("Permutation importance can be expensive; running a small diagnostic sample.")
    sample_n = min(2000, len(X_test))
    X_perm = X_test.iloc[:sample_n].copy()
    y_perm = y_test.iloc[:sample_n].copy()
    perm = permutation_importance(
        final_model, X_perm, y_perm,
        scoring="f1_macro", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1
    )
    perm_imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
    display(perm_imp.rename("permutation_importance").to_frame())


## 11. Save the trained model and final predictions

The saved model includes preprocessing and classification. Predictions are exported with the timestamp, actual label, and predicted label for review. The output files are generated in the current notebook directory.


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

model_path = OUTPUT_DIR / "load_type_classifier.joblib"
predictions_path = OUTPUT_DIR / "test_predictions.csv"
metrics_path = OUTPUT_DIR / "test_metrics.csv"

joblib.dump(final_model, model_path)

prediction_output = test_df[[DATETIME_COL]].copy()
prediction_output["actual_Load_Type"] = y_test.to_numpy()
prediction_output["predicted_Load_Type"] = y_test_pred
prediction_output.to_csv(predictions_path, index=False)

pd.DataFrame([final_metrics]).to_csv(metrics_path, index=False)

print("Saved model:", model_path.resolve())
print("Saved predictions:", predictions_path.resolve())
print("Saved metrics:", metrics_path.resolve())
display(prediction_output.head(10))


## 12. Results and conclusion

Use the actual metrics printed by the notebook in your submission. Do not claim a score before running the notebook on the supplied dataset.

Complete the summary below after execution:
- Selected model: value printed in Section 8
- Test period: value printed in Section 9
- Test accuracy: value in Section 9
- Macro precision: value in Section 9
- Macro recall: value in Section 9
- Macro F1-score: value in Section 9

**Interpretation:** Accuracy measures overall correctness. Macro precision, recall, and F1 treat each class equally, which is useful when class frequencies are imbalanced. The final test set is the last calendar month, in accordance with the assignment instructions.
